# Market Differentiated Bars

## Problem Definition

**Question.** Does the stored fractional log-price feature improve stationarity while retaining observable price information?

**Role in the workflow.** Validate the event-start price feature used by the primary model.

**Inputs.** Local dollar bars and the local fixed-width fractionally differentiated feature Parquet.

**Outputs.** A joined diagnostic table and the validated fractional-feature path.

**Why this method.** Fixed-width fractional differentiation targets the AFML stationarity-memory tradeoff without inventing a price path.

**Assumptions.** The stored differentiation order was estimated upstream; this notebook validates alignment and diagnostics rather than selecting it on the holdout.

**Handoff.** The fractional feature path to `event_labeling.ipynb`.


## Real Data, Development Diagnostic, and Preprocessing

Only observed dollar-bar closes are used. The stationarity diagnostic is descriptive and does not use event labels or the future holdout outcome.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.tsa.stattools import adfuller

PROJECT_ROOT = Path.cwd().resolve().parents[1]
feature_dir = PROJECT_ROOT / "data/research_data/market/features"
period = "2025-01-01_2025-12-31"
dollar_path = feature_dir / f"aapl_dollar_bar_{period}.parquet"
fractional_path = feature_dir / f"aapl_dollar_bar_fractional_{period}.parquet"

dollar_bars = pd.read_parquet(dollar_path)[["end", "close"]].drop_duplicates("end", keep="last")
fractional = pd.read_parquet(fractional_path).drop_duplicates("end", keep="last")
joined = dollar_bars.merge(fractional, on="end", how="inner", validate="one_to_one")
joined["log_close"] = np.log(joined["close"].astype(float))

diagnostic_sample = joined.tail(min(20_000, len(joined)))
diagnostic = pd.Series(
    {
        "dollar_bar_rows": len(dollar_bars),
        "fractional_rows": len(fractional),
        "aligned_rows": len(joined),
        "fractional_missing": int(joined["fractionally_differenced_log_close"].isna().sum()),
        "level_adf_pvalue": float(adfuller(diagnostic_sample["log_close"], maxlag=1, autolag=None)[1]),
        "fractional_adf_pvalue": float(adfuller(diagnostic_sample["fractionally_differenced_log_close"], maxlag=1, autolag=None)[1]),
        "level_fractional_correlation": float(diagnostic_sample[["log_close", "fractionally_differenced_log_close"]].corr().iloc[0, 1]),
    },
    name="value",
)
display(diagnostic.to_frame())
display(joined.head())


## Results, Limitations, and Handoff

An ADF result is sample- and lag-sensitive and does not prove a stable data-generating process. The feature is retained because it is aligned to completed bars and was selected before any model holdout evaluation.

The next notebook receives the aligned fractional log-price feature. No conclusion in this notebook is evidence of live-trading profitability.
